# HSLS:09 descriptives

Erik, 2026-11-18

This notebook describes the cleaned HSLS:09 file before any model uses it: each
variable's type, summary statistics, and distribution, then its missing values,
why they are missing, and how they occur together. `docs/codebook.md` lists every
value and reserve code. `docs/missing-data.md` explains the codes.

Counts are unweighted. Means and percentages are given both unweighted, over the
students in the file, and weighted to the population of 9th graders in fall 2009.

The notebook reads private data, so it renders to `outputs/private/notebooks`.
It shows summaries only, never individual records.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

fw.scaffold()
import hsls  # functions/hsls.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_rows", 200)

crosswalk = fw.data_read("inputs.public.raw.hsls_crosswalk", col_types="c" * 14).fillna("")
clean = fw.data_read("inputs.private.final.hsls_clean", dtype={"student_id": str})
raw = fw.data_read(
    "inputs.private.raw.hsls_student",
    usecols=list(dict.fromkeys(crosswalk["nces"])),
    dtype={"STU_ID": str},
)

# Missing-value reasons compare raw and clean values row by row
assert raw["STU_ID"].tolist() == clean["student_id"].tolist()
reasons = hsls.missing_reasons(raw, clean, crosswalk)
variables = crosswalk.set_index("name")
labels = variables["label"]
f"{len(clean):,} students, {clean.shape[1]} variables"

## Variables

Each variable's type comes from the crosswalk. The stored type is how the file
holds it: every variable except the student ID is numeric, and categories are
codes with labels in the crosswalk.

In [ ]:
overview = pd.DataFrame(
    {
        "Label": labels,
        "Wave": variables["wave"],
        "Type": variables["type"],
        "Stored as": clean.dtypes.astype(str),
        "Distinct values": clean.nunique(),
        "Observed": clean.notna().sum(),
        "Missing (%)": (clean.isna().mean() * 100).round(1),
    }
)
overview

## Sample

HSLS:09 sampled 23,503 9th graders in fall 2009. A weight above 0 marks a
student who counts toward estimates with that weight: the base-year student
weight needs a base-year student survey, the parent weight a parent survey, and
the panel weight both student surveys.

In [ ]:
flags = ["w1_responded", "w1_parent_responded", "w2_responded"]
weights = ["w1_weight", "w1_weight_parent", "w2_weight_panel"]
sample = pd.DataFrame(
    {
        "Label": labels[flags + weights].values,
        "Students": [int((clean[name] == 1).sum()) for name in flags]
        + [int((clean[name] > 0).sum()) for name in weights],
    },
    index=[*(f"{name} = 1" for name in flags), *(f"{name} > 0" for name in weights)],
)
sample["Share (%)"] = (sample["Students"] / len(clean) * 100).round(1)
sample

In [ ]:
# Who answered which survey. Missing means the response status itself is unknown.
pd.crosstab(
    clean["w1_responded"].map({0: "No base-year survey", 1: "Base-year survey"}),
    clean["w2_responded"].map({0: "No follow-up survey", 1: "Follow-up survey"}).fillna("Status unknown"),
    margins=True,
    margins_name="Total",
)

## Continuous variables

The weighted columns use the weight that matches the variable's source: the
base-year student weight for base-year variables, and the panel weight for the
first follow-up math score. Weighted SDs are population SDs.

In [ ]:
continuous = crosswalk.loc[crosswalk["type"] == "continuous", "name"].tolist()
weight_for = {name: "w2_weight_panel" if variables.at[name, "wave"] == "2" else "w1_weight" for name in continuous}

rows = {}
for name in continuous:
    x = clean[name]
    mean, sd = hsls.weighted_mean_sd(x, clean[weight_for[name]])
    rows[name] = {
        "Observed": int(x.notna().sum()),
        "Mean": x.mean(),
        "SD": x.std(),
        "Min": x.min(),
        "5%": x.quantile(0.05),
        "25%": x.quantile(0.25),
        "Median": x.median(),
        "75%": x.quantile(0.75),
        "95%": x.quantile(0.95),
        "Max": x.max(),
        "Weight": weight_for[name],
        "Weighted mean": mean,
        "Weighted SD": sd,
    }
summary = pd.DataFrame(rows).T
numbers = summary.columns.drop("Weight")
summary[numbers] = summary[numbers].astype(float).round(2)
summary

The math scores are T-scores, scaled by NCES to a weighted mean of 50 and SD of
10 in each wave. SES and math self-efficacy are standardized composites, so
their negative values are real values, not reserve codes.

In [ ]:
shown = ["w1_math", "w2_math", "w1_math_efficacy", "w1_ses"]
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))
for name, ax in zip(shown, axes.flat):
    x = clean[name].dropna()
    sns.histplot(x, bins=40, ax=ax)
    ax.set_title(f"{labels[name]} (n = {len(x):,})", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("Students")
fig.suptitle("Distributions of the continuous variables, observed values")
fig.tight_layout()

## Categorical variables

Percentages are of the students with an observed value. The weighted
percentages use the base-year student weight, or the parent weight for
variables from the parent survey.

In [ ]:
# The response flags are in the sample table above
categorical = [
    name
    for name in crosswalk.loc[crosswalk["type"].isin(["binary", "categorical", "ordinal"]), "name"]
    if name not in flags
]
parent_items = [name for name in categorical if variables.at[name, "nces"].startswith(("X1PAR", "X1P2", "X1FAM"))]


def value_label(name, value):
    row = variables.loc[name]
    if row["derive"] == "applies":
        return {0: "Logical skip in the source variable", 1: "Answered"}[value]
    named = hsls.value_labels(row["labels"])
    if not named and row["type"] == "binary":
        named = {0: "No", 1: "Yes"}
    return named.get(value, "")


tables = []
for name in categorical:
    x = clean[name]
    weight = clean["w1_weight_parent" if name in parent_items else "w1_weight"]
    counts = x.value_counts().sort_index()
    weighted = weight[weight > 0].groupby(x).sum().reindex(counts.index, fill_value=0)
    tables.append(
        pd.DataFrame(
            {
                "Variable": name,
                "Value": counts.index.astype(int),
                "Label": [value_label(name, value) for value in counts.index],
                "Students": counts.values,
                "Percent": (counts / counts.sum() * 100).round(1).values,
                "Weighted percent": (weighted / weighted.sum() * 100).round(1).values,
            }
        )
    )
frequencies = pd.concat(tables).set_index(["Variable", "Value"])
frequencies

## Missing values by reason

Each missing value has one of three reasons, from the raw NCES codes:

- **Nonresponse**: the question applied and went unanswered. An analysis
  imputes these or models them.
- **Logical skip**: the question did not apply. An analysis of that variable
  usually drops these students.
- **Outside the flag**: the raw file has an answer, but the variable applies
  only where a flag is 1, such as parent 2's employment where there is no
  second parent.

In [ ]:
order = ["Observed", "Nonresponse", "Logical skip", "Outside the flag"]
by_reason = reasons.apply(lambda column: column.value_counts()).T.reindex(columns=order).fillna(0).astype(int)
by_reason.insert(0, "Label", labels)
by_reason

In [ ]:
missing_share = by_reason[order[1:]].div(len(clean)).mul(100)
missing_share = missing_share[missing_share.sum(axis=1) > 0].sort_values("Nonresponse")
ax = missing_share.plot.barh(stacked=True, figsize=(8, 7), width=0.8)
ax.set_xlabel("Students (%)")
ax.set_ylabel("")
ax.set_title("Missing values by reason, variables with any missing")
ax.legend(title="", loc="lower right")
plt.tight_layout()

## Missingness patterns

The patterns below use the variables a first model is likely to use: the two
math scores, SES, math self-efficacy, sex, race, parents' education, and family
income. Each row is a combination of observed (`·`) and missing (`X`) values,
counting a missing value for any reason.

In [ ]:
analysis = [
    "w1_math",
    "w2_math",
    "w1_ses",
    "w1_math_efficacy",
    "w1_female",
    "w1_race",
    "w1_parents_edu",
    "w1_family_income",
]
missing = clean[analysis].isna()
patterns = (
    missing.replace({True: "X", False: "·"})
    .value_counts()
    .rename("Students")
    .reset_index()
)
patterns["Percent"] = (patterns["Students"] / len(clean) * 100).round(1)
patterns["Missing variables"] = patterns[analysis].eq("X").sum(axis=1)
patterns.head(15)

In [ ]:
complete = int((~missing.any(axis=1)).sum())
print(f"{complete:,} students ({complete / len(clean):.1%}) have all {len(analysis)} variables.")
print(f"{len(patterns)} distinct patterns. The 15 above cover {patterns['Students'].head(15).sum() / len(clean):.1%} of students.")

Missing values cluster by survey. A student without a base-year survey misses
the base-year score, SES, and self-efficacy together. A parent who did not
respond leaves parents' education and income missing. The next table gives the
share missing on each variable within each response group.

In [ ]:
def share_missing(flag):
    groups = clean[flag].map({0: "No", 1: "Yes"}).fillna("Unknown")
    table = missing.groupby(groups).mean().mul(100).round(1).T
    table.columns = [f"{flag} = {group}" for group in table.columns]
    return table


pd.concat([share_missing(flag) for flag in flags], axis=1)

In [ ]:
# Of students missing the row variable, the share also missing the column variable
both = missing.T.astype(int) @ missing.astype(int)
conditional = both.div(missing.sum(), axis=0).mul(100)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(conditional, annot=True, fmt=".0f", cmap="viridis", vmin=0, vmax=100, cbar_kws={"label": "Percent"}, ax=ax)
ax.set_xlabel("Also missing")
ax.set_ylabel("Missing")
ax.set_title("Share missing the column variable, among students missing the row variable")
plt.tight_layout()

## Missingness and observed values

If the students missing a value differ on what is observed, a complete-case
analysis describes a different group than the sample. The table compares
students with and without each outcome on base-year values that are observed
for both groups. The standardized difference is the difference in means divided
by the pooled SD. Values beyond about 0.1 are usually read as a meaningful
imbalance.

In [ ]:
def compare(missing_on, covariates):
    is_missing = clean[missing_on].isna()
    rows = {}
    for name in covariates:
        observed = clean.loc[~is_missing, name].dropna()
        absent = clean.loc[is_missing, name].dropna()
        pooled = np.sqrt((observed.var() + absent.var()) / 2)
        rows[name] = {
            f"Mean, {missing_on} observed": observed.mean(),
            f"Mean, {missing_on} missing": absent.mean(),
            "Standardized difference": (absent.mean() - observed.mean()) / pooled,
            "n missing with a value": len(absent),
        }
    return pd.DataFrame(rows).T.round(2)


display(compare("w2_math", ["w1_math", "w1_ses", "w1_math_efficacy", "w1_female", "w1_parents_edu"]))
display(compare("w1_parents_edu", ["w1_math", "w1_math_efficacy", "w1_female"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
groups = clean["w2_math"].isna().map({False: "Follow-up score observed", True: "Follow-up score missing"})
sns.kdeplot(x=clean["w1_math"], hue=groups, common_norm=False, ax=ax)
ax.set_xlabel(labels["w1_math"])
ax.set_ylabel("Density")
ax.set_title("Base-year math score by whether the follow-up score is observed")
ax.get_legend().set_title("")
plt.tight_layout()

## Relationships

Correlations use all students with both values, so each pair can rest on a
different group. The table after the matrix gives each pair's count.

In [ ]:
related = ["w1_math", "w2_math", "w1_math_efficacy", "w1_ses", "w1_parents_edu", "w1_family_income"]
display(clean[related].corr().round(2))
observed = clean[related].notna().astype(int)
observed.T @ observed

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
pair = clean[["w1_math", "w2_math"]].dropna()
ax.hexbin(pair["w1_math"], pair["w2_math"], gridsize=45, cmap="viridis", mincnt=1)
ax.set_xlabel(labels["w1_math"])
ax.set_ylabel(labels["w2_math"])
ax.set_title(f"Math scores in the two waves (n = {len(pair):,})")
plt.tight_layout()

## SES imputations

For a student whose parent did not respond, `w1_ses` is the mean of five
imputations. The five spread around that mean, and averaging them removes the
spread. The table shows, by imputation status, the SD across the five
imputations within a student, averaged over students, and the SD of `w1_ses`
and of the first imputation.

In [ ]:
imputations = [f"w1_ses_imp{i}" for i in range(1, 6)]
status = clean["w1_ses_imputed"].map(
    {0: "0: all components reported", 1: "1: parent did not respond", 2: "2: some components imputed"}
)
pd.DataFrame(
    {
        "Students": status.value_counts(),
        "SD across imputations, mean": clean[imputations].std(axis=1).groupby(status).mean(),
        "Mean minus imputation mean, largest": (clean["w1_ses"] - clean[imputations].mean(axis=1)).abs().groupby(status).max(),
        "SD of w1_ses": clean["w1_ses"].groupby(status).std(),
        "SD of imputation 1": clean["w1_ses_imp1"].groupby(status).std(),
    }
).round(3)

## Summary

- **Weights.** The base-year math score has a weighted mean of 50 and SD of 10,
  as NCES scaled it. The unweighted mean is about 51, so the sample leans
  toward higher scorers.
- **Complete cases.** 56.2% of students have all eight analysis variables. Most
  of the rest miss a whole survey: the base-year student survey, the parent
  survey, or the follow-up.
- **Who is missing.** Students without a follow-up math score scored 0.44 SD
  lower in the base year and had SES 0.29 SD lower than students with one.
  Students missing parents' education, nearly all because the parent did not
  respond, scored 0.44 SD lower too. A
  complete-case analysis drops lower-scoring, lower-SES students, so the
  models in this series need to address missing values rather than drop them.
- **SES.** Among students whose parent did not respond, `w1_ses` has an SD of
  0.54, against 0.82 for a single imputation. The five imputations differ by
  an average SD of 0.66 within a student. Treating `w1_ses` as observed
  ignores that uncertainty.